# Epsilon Fund — CPCV Portfolio Analysis

Combines per-asset CPCV results into a portfolio-level return distribution
via random path sampling. Companion to the individual asset CPCV notebooks.

Run **after** completing the per-asset CPCV notebooks so each `*_cpcv.pkl`
file exists. Load them here, sample portfolio paths, and interpret the
overlap-adjusted confidence intervals at the portfolio level.

---
### What portfolio CPCV gives you

| Output | Meaning |
|--------|---------|
| **Sampled portfolio paths** | N random combinations of per-asset paths — each covers the full dataset once |
| **Portfolio distribution** | Sharpe / Calmar / MaxDD distribution across all sampled combinations |
| **Overlap-adjusted CI** | N_eff-corrected confidence intervals (paths share CPCV splits → not independent) |
| **Per-asset heatmaps** | Which group-pair splits drove strong or weak portfolio paths for each asset |
| **Asset vs portfolio** | Box-plot comparison of individual asset vs combined portfolio Sharpe ranges |

---
### Workflow

1. Run the per-asset CPCV notebooks to generate `*_cpcv.pkl` files.
2. Set `WEIGHTS` and `PKL_PATHS` in the configuration cells below.
3. Optionally fill in `WF_SHARPES` from the walk-forward notebooks.
4. Run all cells top-to-bottom.
5. Save the portfolio paths and CI results for downstream use.

---
### Interpreting results

| Signal | Meaning | Action |
|--------|---------|--------|
| Portfolio Sharpe mean > asset medians | Diversification benefit is real | Confirm with overlap-adjusted CI |
| N_eff ≈ 1 | Paths share almost every CPCV split — effectively one path | Trust the adjusted CI, not the naive CI |
| Wide adjusted CI | High path-to-path variance after correlation correction | Collect more assets or increase N_SAMPLES |
| Heatmap shows consistent red group pairs | Those time periods are structurally hard for every asset | Consider excluding them or reducing exposure |

---
## Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from cpcv_portfolio import (
    load_asset_cpcv,
    sample_portfolio_paths,
    portfolio_confidence_intervals,
    portfolio_summary,
    per_asset_split_heatmaps,
    diversification_benefit,
    asset_correlation_structure,
    worst_drawdown_decomposition,
    optimal_weights,
    extract_portfolio_trade_stats,
    trade_stats_confidence_intervals,
    trade_stats_summary,
    per_asset_yearly_breakdown,
    build_representative_oos_dfs,
)
from cpcv_portfolio_visualizer import (
    plot_portfolio_equity_curves,
    plot_portfolio_distribution,
    plot_per_asset_heatmaps,
    plot_portfolio_vs_assets,
    plot_portfolio_full_results,
    plot_diversification_benefit,
    plot_correlation_structure,
    plot_drawdown_decomposition,
    plot_weight_optimisation,
    plot_yearly_performance,
    plot_trade_statistics,
    plot_asset_yearly_heatmap,
)
from wf_visualizer import plot_closed_trade_equity
from cpcv_engine import cpcv_summary

---
## Configuration

In [ ]:
# ── portfolio weights ─────────────────────────────────────────────────────────
# None  → equal weight across all loaded assets (default).
# Override with a dict that sums to 1.0 to apply custom weights, e.g.:
#   WEIGHTS = {'BTC': 0.40, 'ETH': 0.35, 'SOL': 0.25}
WEIGHTS = None

# ── sampling ──────────────────────────────────────────────────────────────────
N_SAMPLES = 2000   # random portfolio path samples
SEED      = 42     # RNG seed — fix for reproducibility

# ── walk-forward Sharpes for reference overlay (optional) ────────────────────
# Add entries for any asset whose WF Sharpe you want drawn as a reference vline.
# Leave empty to skip the overlay entirely.
WF_SHARPES = {
    # 'BTC': 1.45,   # ← fill in from walk-forward notebook
}

---
## Load Per-Asset CPCV Results

> **Adjust paths if pkl files are in a different directory.**  
> Run the per-asset CPCV notebooks first — each saves a `*_cpcv.pkl` produced by
> `run_cpcv()`. The default naming convention from those notebooks is
> `{symbol.lower()}_cpcv.pkl` (e.g. `ethusdt_cpcv.pkl`).  
>  
> **All assets must share the same N, n\_splits, and n\_paths** — `load_asset_cpcv`
> will raise a `ValueError` on any structural mismatch. Different calendar date
> ranges are fine and expected; paths are aligned via inner-join on dates inside
> `sample_portfolio_paths`.

In [ ]:
# ── pkl paths ─────────────────────────────────────────────────────────────────
# Paths are relative to the notebook directory.  Use absolute paths if needed:
#   os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'momentum_cpcv', 'ethusdt_cpcv.pkl')
PKL_PATHS = {
    'ADA':  'oos/adausdt_cpcv.pkl',
    'AVAX': 'oos/avaxusdt_cpcv.pkl',
   # 'BNB':  'oos/bnbusdt_cpcv.pkl',
    'BTC':  'oos/btcusdt_cpcv.pkl',
    'ETH':  'oos/ethusdt_cpcv.pkl',
    'SOL':  'oos/solusdt_cpcv.pkl',
    'XRP':  'oos/xrpusdt_cpcv.pkl',
}

asset_results = load_asset_cpcv(PKL_PATHS)

# ── compute equal weights if not overridden ───────────────────────────────────
if WEIGHTS is None:
    _n      = len(asset_results)
    _w      = 1.0 / _n
    WEIGHTS = {asset: _w for asset in asset_results}
    print(f'Equal weights ({_n} assets):')
    for asset, w in WEIGHTS.items():
        print(f'  {asset:<6} {w * 100:.4f}%')
else:
    print('Custom weights:')
    for asset, w in WEIGHTS.items():
        print(f'  {asset:<6} {w * 100:.4f}%')

---
## Individual Asset Results (reference)

One-line summary per asset before combining — confirm each asset's CPCV is healthy
before interpreting portfolio-level results.

In [ ]:
for asset, results in asset_results.items():
    path_sharpes = [
        p['sharpe'] for p in results['paths']
        if p.get('sharpe') is not None
    ]
    median_sh = float(np.median(path_sharpes)) if path_sharpes else float('nan')
    n_paths   = len(path_sharpes)
    print(f'{asset}: median path Sharpe = {median_sh:.2f}, N paths = {n_paths}')

---
## Portfolio Path Sampling

For each of `N_SAMPLES` draws, one valid CPCV path is independently sampled per
asset. Returns are inner-joined on the common date index and combined using
`WEIGHTS`. Metrics (Sharpe, Calmar, MaxDD) are computed on the weighted portfolio
return series using the same formulas as the backtester.

| Parameter | Description |
|-----------|-------------|
| `N_SAMPLES` | Random combinations to draw — more samples reduce sampling noise in the distribution |
| `SEED` | RNG seed — fix at 42 for reproducibility across runs |

In [ ]:
portfolio_paths = sample_portfolio_paths(
    asset_results, WEIGHTS,
    n_samples=N_SAMPLES, seed=SEED,
)
print(f'Sampled {len(portfolio_paths)} portfolio paths')

---
## Portfolio Summary

Three tables:

1. **Path distribution** — Mean / Std / Min / Q25 / Median / Q75 / Max / IQR across
   Sharpe, Calmar, Max DD, and Return for all sampled portfolio paths.
2. **Confidence intervals** — naive (anticonservative, assumes independent paths) and
   overlap-adjusted (conservative, accounts for shared CPCV splits via N_eff).
   The **conservative lower bound** is the most cautious point estimate of mean Sharpe.
3. **Asset weights** — per-asset weight, median individual Sharpe, and normalised
   contribution (weight × median Sharpe).

> **Note on N_eff:** With multiple assets each contributing 28 splits, almost every
> pair of portfolio paths shares at least one split across some asset, collapsing N_eff
> close to 1. This is correct — the adjusted CI is wide and conservative. Use the
> conservative lower bound as your hurdle-rate benchmark.

In [ ]:
ci = portfolio_confidence_intervals(portfolio_paths, asset_results)
portfolio_summary(portfolio_paths, ci, WEIGHTS, asset_results=asset_results)

---
## Portfolio Path Distribution

Three charts produced by `plot_portfolio_results`:

- **Equity curves** — up to 200 sampled portfolio equity paths (blue, semi-transparent),
  mean path (amber), and min–max envelope. All paths are base-100.
- **Sharpe histogram** — distribution of sampled portfolio Sharpe ratios. The blue
  shaded band is the overlap-adjusted 95% CI; the dashed red line is the conservative
  floor. Dotted coloured vlines show each asset's individual CPCV median Sharpe.
- **Asset vs portfolio box plots** — per-asset CPCV path Sharpe distributions (blue)
  sorted by median, plus the sampled portfolio distribution (amber).

In [ ]:
# Filter out None WF_SHARPES entries so the vline call doesn't fail
_wf = {k: v for k, v in WF_SHARPES.items() if v is not None}

plot_portfolio_full_results(
    portfolio_paths,
    asset_results,
    ci_results=ci,
    weights=WEIGHTS,
    wf_sharpes=_wf if _wf else None,
);

---
## Closed-Trade Equity

Same underlying strategy and **the same closed-trade-only sizing model** as the path
distribution above — position notional is scaled by realized (closed) equity only;
unrealized gains never inflate future trade sizes in either chart.

The only difference is **temporal resolution**:

| Chart | Curve moves | Paths |
|---|---|---|
| CPCV path distribution (above) | Every bar — intra-trade fluctuations visible | 2000 random path combinations |
| Closed-trade equity (below) | Only on trade exits — flat between trades | One representative path per asset |

Each asset uses its **representative path**: the CPCV path whose Sharpe is closest
to that asset's median across all 105 paths. The weighted average of per-asset
exit-by-exit returns produces the portfolio closed-trade curve.

> The closed-trade view isolates trade-by-trade realized P&L with no intra-trade noise.  
> Read it alongside the path distribution, not instead of it.

In [ ]:
COST_CT = 0.001   # keep aligned with cost used in plot_portfolio_full_results above

rep_dfs = build_representative_oos_dfs(asset_results, WEIGHTS)

plot_closed_trade_equity(
    rep_dfs,
    weights = WEIGHTS,
    cost    = COST_CT,
);

---
## Per-Asset Group Pair Analysis

For each asset, a heatmap showing the mean **portfolio** Sharpe across all sampled
paths where that asset used a given (group_i, group_j) CPCV split.

- **Green** cell — when this asset's path was drawn from that group pair, the sampled
  portfolio paths tended to have higher Sharpe.
- **Red** cell — those time periods are structurally weak for the portfolio, regardless
  of which other assets' paths were drawn.

Colorscale is centred at the global mean across all assets and pairs — not at zero —
so green/red reflects above/below-average portfolio performance.

Cross-reference each group's calendar window with `cpcv_summary(..., show_group_legend=True)`
in the per-asset CPCV notebooks to identify which market regime each group covers.

In [ ]:
heatmap_data = per_asset_split_heatmaps(portfolio_paths, asset_results)
plot_per_asset_heatmaps(heatmap_data, asset_results);

---
## Diversification Benefit

Measures the Sharpe **uplift** from combining assets versus a weighted basket of
individual strategies:

    diversification_benefit = portfolio_sharpe − Σ(weight_i × asset_sharpe_i)

- **Positive** → correlation structure reduces portfolio volatility below the
  weighted-average of individual strategy vols.
- **Negative** → assets are too correlated to diversify meaningfully at this path.
- The histogram splits green (positive benefit) / red (negative); the scatter shows
  weighted-average component Sharpe vs portfolio Sharpe coloured by benefit magnitude.

In [ ]:
div_benefit = diversification_benefit(portfolio_paths, asset_results, WEIGHTS)

plot_diversification_benefit(div_benefit, WEIGHTS);

---
## Asset Correlation Structure

Computes pairwise Pearson return correlations across all sampled portfolio paths.
Each path gives one correlation estimate per asset pair; the distribution across
paths reveals how stable (or regime-dependent) correlations are.

- **Violin per pair** — full distribution of per-path correlations; pairs coloured
  by median: green < 0.3, amber < 0.5, red ≥ 0.5. Dashed line at 0.7 flags
  high-correlation paths.
- **Heatmap** — symmetric matrix of mean pairwise correlations (±std in cell text).

In [ ]:
corr_results = asset_correlation_structure(
    portfolio_paths,
    asset_results,
    WEIGHTS,
)

plot_correlation_structure(corr_results);

---
## Worst Drawdown Decomposition

Identifies the `n_worst` portfolio paths by maximum drawdown and decomposes each
drawdown period into per-asset contributions.

- **Contribution** = `weight × cumulative return over the drawdown window`
- **Pct of DD** = asset contribution as a fraction of the total portfolio drawdown
- **Primary driver** = asset with the highest mean `|pct_of_dd|` across worst paths

Use this to understand which assets dominate portfolio risk in tail scenarios.

In [ ]:
dd_results = worst_drawdown_decomposition(
    portfolio_paths,
    asset_results,
    WEIGHTS,
    worst_pct=0.10,
)

plot_drawdown_decomposition(dd_results, WEIGHTS);

---
## Per-Asset Yearly Performance

For each calendar year, shows how each asset contributed to portfolio returns
across all sampled paths.

- **Top heatmap** — mean annual return per asset (RdYlGn, diverging at 0). The
  bottom row is the portfolio aggregate. Cell text is formatted as `±XX.X%`.
- **Middle heatmap** — fraction of paths where the asset had a positive year
  (RdYlGn, midpoint at 50%). Cells below 50% are shaded red.
- **Bottom heatmap** — asset rank by mean return within each year (rank 1 = best).
  Green = top-ranked; red = bottom-ranked. Useful for spotting rotation patterns.

In [ ]:
yearly = per_asset_yearly_breakdown(portfolio_paths, asset_results, WEIGHTS)

# Quick pivot: mean annual return per asset × year
print(yearly['asset_year_summary']
      .pivot(index='asset', columns='year', values='mean_return')
      .applymap(lambda v: f'{v*100:.1f}%')
      .to_string())

plot_asset_yearly_heatmap(yearly);

pd.to_pickle(yearly, 'oos/portfolio_cpcv_yearly.pkl')

---
## Trade Statistics

Extracts trade-level data from the per-asset OOS strategy DataFrames stored
inside each CPCV split result.  For every sampled portfolio path the following
are aggregated across all assets and all CPCV groups assigned to that path:

| Statistic | Description |
|-----------|-------------|
| **Win rate** | Fraction of trades with positive PnL |
| **Avg trade return** | Mean PnL across all trades |
| **Profit factor** | Gross profit / gross loss |
| **Avg holding period** | Mean bars per trade |
| **Trades per year** | Annualised trade count |
| **Consecutive runs** | Max consecutive wins / losses |

Yearly stats (return, Sharpe, MaxDD, trade count) are computed from
`portfolio_returns` and the per-group trade DataFrames.

In [ ]:
trade_stats = extract_portfolio_trade_stats(portfolio_paths, asset_results, WEIGHTS)

trade_cis = trade_stats_confidence_intervals(trade_stats, portfolio_paths, asset_results)

trade_stats_summary(trade_stats, trade_cis)

In [ ]:
plot_trade_statistics(trade_stats, trade_cis);
plot_yearly_performance(trade_stats, trade_cis);

---
## Optimal Weight Search

Samples `N_WEIGHT_CONFIGS` random weight configurations using Dirichlet sampling
(clipped to [5%, 60%] per asset) and evaluates each via `N_SAMPLES` portfolio paths.
Three objectives are reported:

| Objective | Meaning |
|-----------|---------|
| `max_conservative_sharpe` | Maximise the overlap-adjusted conservative Sharpe lower bound |
| `max_mean_sharpe` | Maximise mean portfolio Sharpe across all sampled paths |
| `min_max_dd` | Minimise mean maximum drawdown |

The equal-weight allocation is always evaluated as config 0 for reference.

In [ ]:
N_WEIGHT_CONFIGS = 500  # random weight configurations to evaluate

optimal_results = optimal_weights(
    asset_results,
    n_samples        = 300,   # wider CI but still meaningful
    n_weight_configs = 150,   # 200 random weight vectors
)

plot_weight_optimisation(optimal_results, list(WEIGHTS.keys()));

---
## Save All Results

Pickle all portfolio-level outputs for downstream use (e.g. comparing multiple
portfolio configurations, building a combined report, or re-loading results
without re-running the full analysis).

In [ ]:
os.makedirs('oos', exist_ok=True)
pd.to_pickle(portfolio_paths,  'oos/portfolio_cpcv_paths.pkl')
pd.to_pickle(ci,               'oos/portfolio_cpcv_ci.pkl')
pd.to_pickle(div_benefit,      'oos/portfolio_cpcv_div_benefit.pkl')
pd.to_pickle(corr_results,     'oos/portfolio_cpcv_corr.pkl')
pd.to_pickle(dd_results,       'oos/portfolio_cpcv_dd.pkl')
pd.to_pickle(optimal_results,  'oos/portfolio_cpcv_weights.pkl')
pd.to_pickle(trade_stats,      'oos/portfolio_cpcv_trade_stats.pkl')
pd.to_pickle(trade_cis,        'oos/portfolio_cpcv_trade_cis.pkl')
pd.to_pickle(yearly,           'oos/portfolio_cpcv_yearly.pkl')
print("All portfolio CPCV results saved → oos/")